### Load Data Files into Polars DataFrame

In [ ]:
from google.colab import drive
from typing import Literal
from pathlib import Path
import os
import polars as pl

# config polars
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(-1)

# mount drive
drive.mount('/content/drive')
path = Path('/content/drive/MyDrive/hydromind')
data_path = Path(os.path.join(path, "data"))
hemisphere = Literal["north", "south"]

def load_data(hemisphere: str) -> dict[str, pl.DataFrame]:
    # load all data for one hemisphere
    return {
        "household": pl.read_csv(
            data_path / f"{hemisphere}_household.csv",
            dtypes={
                "household_id": pl.Utf8,
                "occupancy_count": pl.Int16,
                "appliance_efficiency_score": pl.Float32,
                "landscape_type": pl.Categorical
            },
        ),
        "environment": pl.read_csv(
            data_path / f"{hemisphere}_environment.csv",
            dtypes={
                "daily_max_temp_celsius": pl.Float32,
                "daily_rainfall_mm": pl.Float32
            },
        ),
        "water_usage": pl.read_parquet(
            data_path / f"{hemisphere}_water_usage.parquet"
        ).cast({
            col: pl.Float32 for col in [str(i) for i in range(365)]
        })
    }

df_north = load_data(hemisphere="north")
df_south = load_data(hemisphere="south")

### Inspect First 10 Rows in Household Data

In [ ]:
print(df_north["household"].head(10))
print(df_south["household"].head(10))

### Inspect First 10 Rows in Environment Data

In [ ]:
print(df_north["environment"].head(10))
print(df_south["environment"].head(10))

### Inspect First 10 Rows in Water Usage Data

In [ ]:
print(df_north["water_usage"].head(10))
print(df_south["water_usage"].head(10))

### Data Profiling and Quality Checks

In [ ]:
def profile_data(df: pl.DataFrame) -> pl.DataFrame:
    # generate profiling report: shape, nulls, dtypes, unique counts
    metrics = {
        "column": [], "dtype": [], "null_count": [], "n_unique": [], 
        "min": [], "max": [], "mean": [], "median": [], 
        "mode": [], "variance": [], "std": []
    }
    
    for col_name, dtype in df.schema.items():
        # extract the series
        s = df[col_name]
        is_num = dtype.is_numeric()
        
        # append base metrics
        metrics["column"].append(col_name)
        metrics["dtype"].append(str(dtype))
        metrics["null_count"].append(s.null_count())
        metrics["n_unique"].append(s.n_unique())
        
        # append numeric metrics
        metrics["min"].append(s.min() if is_num else None)
        metrics["max"].append(s.max() if is_num else None)
        metrics["mean"].append(s.mean() if is_num else None)
        metrics["median"].append(s.median() if is_num else None)
        metrics["mode"].append(s.mode().first() if is_num else None) 
        metrics["variance"].append(s.var() if is_num else None)
        metrics["std"].append(s.std() if is_num else None)
        
    # construct the final DataFrame
    return pl.DataFrame(metrics, strict=False)

In [ ]:
# Household Data Profiling
print(profile_data(df_north["household"]))
print(profile_data(df_south["household"]))

In [ ]:
# Environment Data Profiling
print(profile_data(df_north["environment"]))
print(profile_data(df_south["environment"]))

In [ ]:
# Water Usage Data Profiling
print(profile_data(df_north["water_usage"]))
print(profile_data(df_south["water_usage"]))

### Cross-File Integrity Check

In [ ]:
def validate_cross_integrity(data: dict) -> list[str]:
    # ensures dimensional consistency across files
    errors = []
    household_count = data["household"].height
    water_usage_rows = data["water_usage"].height
    environment_days = data["environment"].height

    if household_count != water_usage_rows:
        errors.append(f"Household Count ({household_count}) != Water Usage Rows ({water_usage_rows})")
    if environment_days != 365:
        errors.append(f"Environment Days ({environment_days}) != 365")
    if data["water_usage"].width != 365:
        errors.append(f"Water Usage Columns ({data['water_usage'].width}) != 365")
    
    return errors

north_data_errors = validate_cross_integrity(df_north)
south_data_errors = validate_cross_integrity(df_south)

if (north_data_errors):
    print(north_data_errors)
if (south_data_errors):
    print(south_data_errors)

if not north_data_errors and not south_data_errors:
    print("Ready for the Univariate Analysis!")
